In [1]:
import sys
#sys.path.append('/Users/joachim/texjs/lva/IntroSC/ASC-ODE/build/mechsystem')
sys.path.append('../build/mechsystem')

# Force reimport of the module
import importlib
if 'mass_spring' in sys.modules:
    del sys.modules['mass_spring']

from mass_spring import *
from pythreejs import *

In [ ]:
mss = MassSpringSystem3d()
mss.gravity = (0,0,-9.81)

mA = mss.add (Mass(3, (1,0,0)))
mB = mss.add (Mass(2, (2,0,0)))
f1 = mss.add (Fix( (0,0,0)) )
mss.add_constraint (DistanceConstraint(f1, mA, 1.0))
mss.add_constraint (DistanceConstraint(mA, mB, 1.0))

In [ ]:
masses = []
for m in mss.masses:
    masses.append(
        Mesh(SphereBufferGeometry(0.2, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos)) 

fixes = []
for f in mss.fixes:
    fixes.append(
        Mesh(SphereBufferGeometry(0.08, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos)) 

constraintpos = []
for c in mss.constraints:
    pA = mss[c.connectors[0]].pos
    pB = mss[c.connectors[1]].pos
    constraintpos.append ([ pA, pB ] ) 

springgeo = LineSegmentsGeometry(positions=constraintpos)
m2 = LineMaterial(linewidth=3, color='cyan')
springs = LineSegments2(springgeo, m2)    

axes = AxesHelper(1)

In [ ]:
view_width = 600
view_height = 400

camera = PerspectiveCamera( position=[4, 10, 6], aspect=view_width/view_height)
key_light = DirectionalLight(position=[0, 10, 10])
ambient_light = AmbientLight()
camera.up = [0, 0, 1]
#axes.visible = False

scene = Scene(children=[*masses, *fixes, springs, axes, camera, key_light, ambient_light])
controller = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controller],
                    width=view_width, height=view_height)

renderer

In [ ]:
from time import sleep
for i in range(10000):
    mss.simulate (0.02, 100)
    for m,mvis in zip(mss.masses, masses):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    constraintpos = []
    for c in mss.constraints:
        pA = mss[c.connectors[0]].pos
        pB = mss[c.connectors[1]].pos
        constraintpos.append ([ pA, pB ]) 
    springs.geometry = LineSegmentsGeometry(positions=constraintpos)
    sleep(0.01)

## 5-Point Pendulum Simulation

Now let's create a similar system with 5 masses (points):

In [ ]:
mss5 = MassSpringSystem3d()
mss5.gravity = (0,0,-9.81)

m1 = mss5.add (Mass(1, (1,0,0)))
m2 = mss5.add (Mass(2, (2,0,0)))
m3 = mss5.add (Mass(3, (3,0,0)))
m4 = mss5.add (Mass(4, (4,0,0)))
m5 = mss5.add (Mass(5, (5,0,0)))
f1_5pt = mss5.add (Fix( (0,0,0)) )
mss5.add_constraint (DistanceConstraint(f1_5pt, m1, 1.0))
mss5.add_constraint (DistanceConstraint(m1, m2, 1.0))
mss5.add_constraint (DistanceConstraint(m2, m3, 1.0))
mss5.add_constraint (DistanceConstraint(m3, m4, 1.0))
mss5.add_constraint (DistanceConstraint(m4, m5, 1.0))

In [ ]:
masses5 = []
for m in mss5.masses:
    masses5.append(
        Mesh(SphereBufferGeometry(0.2, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos)) 

fixes5 = []
for f in mss5.fixes:
    fixes5.append(
        Mesh(SphereBufferGeometry(0.2, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos)) 

constraintpos5 = []
for c in mss5.constraints:
    pA = mss5[c.connectors[0]].pos
    pB = mss5[c.connectors[1]].pos
    constraintpos5.append ([ pA, pB ] ) 

springgeo5 = LineSegmentsGeometry(positions=constraintpos5)
m2_5 = LineMaterial(linewidth=3, color='cyan')
springs5 = LineSegments2(springgeo5, m2_5)    

axes5 = AxesHelper(1)

In [ ]:
view_width5 = 600
view_height5 = 400

camera5 = PerspectiveCamera( position=[4, 10, 6], aspect=view_width5/view_height5)
key_light5 = DirectionalLight(position=[0, 10, 10])
ambient_light5 = AmbientLight()
camera5.up = [0, 0, 1]

scene5 = Scene(children=[*masses5, *fixes5, springs5, axes5, camera5, key_light5, ambient_light5])
controller5 = OrbitControls(controlling=camera5)
renderer5 = Renderer(camera=camera5, scene=scene5, controls=[controller5],
                    width=view_width5, height=view_height5)

renderer5

In [ ]:
from time import sleep
for i in range(10000):
    mss5.simulate (0.02, 100)
    for m,mvis in zip(mss5.masses, masses5):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    constraintpos5 = []
    for c in mss5.constraints:
        pA = mss5[c.connectors[0]].pos
        pB = mss5[c.connectors[1]].pos
        constraintpos5.append ([ pA, pB ]) 
    springs5.geometry = LineSegmentsGeometry(positions=constraintpos5)
    sleep(0.01)

## Spinning Top (Kreisel)

A spinning top is a rigid body structure that maintains its shape through distance constraints while rotating with angular momentum:


In [ ]:
# Create a spinning top with 3 masses in a triangular configuration
mss_top = MassSpringSystem3d()
mss_top.gravity = (0, 0, -9.81)

# Pivot point at origin
f_pivot = mss_top.add(Fix((0, 0, 0)))

# Three masses arranged in a triangle in the xy-plane, offset down in z
# They form an equilateral triangle with side length 1.0
import math
r = 0.5  # radius from center to each mass in xy-plane
pivot_dist = 1.0  # distance from pivot to each mass

# Calculate z-offset so that distance from origin = pivot_dist
# sqrt(r^2 + z^2) = pivot_dist  =>  z = -sqrt(pivot_dist^2 - r^2)
z_offset = math.sqrt(pivot_dist**2 - r**2)
mass_val = 1.0

# Mass positions (equilateral triangle centered at (0, 0, z_offset))
# Initial positions satisfy the pivot distance constraint exactly
m_top_1 = mss_top.add(Mass(mass_val, (r, 0, z_offset)))
m_top_2 = mss_top.add(Mass(mass_val, (r*math.cos(2*math.pi/3), r*math.sin(2*math.pi/3), z_offset)))
m_top_3 = mss_top.add(Mass(mass_val, (r*math.cos(4*math.pi/3), r*math.sin(4*math.pi/3), z_offset)))

# Distance constraints: connect pivot to each mass
mss_top.add_constraint(DistanceConstraint(f_pivot, m_top_1, pivot_dist))
mss_top.add_constraint(DistanceConstraint(f_pivot, m_top_2, pivot_dist))
mss_top.add_constraint(DistanceConstraint(f_pivot, m_top_3, pivot_dist))

# Distance constraints: connect masses to each other (rigid triangle)
triangle_dist = math.sqrt(3) * r
mss_top.add_constraint(DistanceConstraint(m_top_1, m_top_2, triangle_dist))
mss_top.add_constraint(DistanceConstraint(m_top_2, m_top_3, triangle_dist))
mss_top.add_constraint(DistanceConstraint(m_top_3, m_top_1, triangle_dist))

# Initialize rotational velocities (angular velocity around z-axis)
omega = 10.0  # rad/s
# For rotation around z: velocity = omega × position = (-omega*y, omega*x, 0)
for i, m in enumerate(mss_top.masses):
    pos = m.pos
    mss_top.set_mass_velocity(i, (-omega * pos[1], omega * pos[0], 0.0))

print("✓ Spinning top created with 3 masses")
print(f"  - 1 pivot point fixed at origin")
print(f"  - 3 masses at radius {r} m in xy-plane, z-offset = {z_offset:.3f} m")
print(f"  - Initial angular velocity: {omega} rad/s around z-axis")
print(f"  - 6 distance constraints maintaining rigid body shape")
print(f"\nConstraints maintain:")
print(f"  - Each mass at distance {pivot_dist:.3f} m from pivot")
print(f"  - Triangle edge length {triangle_dist:.3f} m")

In [ ]:
# Visualize the spinning top
masses_top = []
for m in mss_top.masses:
    masses_top.append(
        Mesh(SphereBufferGeometry(0.1, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos))

fixes_top = []
for f in mss_top.fixes:
    fixes_top.append(
        Mesh(SphereBufferGeometry(0.15, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos))

constraintpos_top = []
for c in mss_top.constraints:
    pA = mss_top[c.connectors[0]].pos
    pB = mss_top[c.connectors[1]].pos
    constraintpos_top.append([pA, pB])

springgeo_top = LineSegmentsGeometry(positions=constraintpos_top)
m2_top = LineMaterial(linewidth=2, color='cyan')
springs_top = LineSegments2(springgeo_top, m2_top)

axes_top = AxesHelper(1)

In [ ]:
# Set up the 3D visualization for the spinning top
view_width_top = 600
view_height_top = 400

camera_top = PerspectiveCamera(position=[2, 2, 2], aspect=view_width_top/view_height_top)
key_light_top = DirectionalLight(position=[0, 10, 10])
ambient_light_top = AmbientLight()
camera_top.up = [0, 0, 1]

scene_top = Scene(children=[*masses_top, *fixes_top, springs_top, axes_top, camera_top, key_light_top, ambient_light_top])
controller_top = OrbitControls(controlling=camera_top)
renderer_top = Renderer(camera=camera_top, scene=scene_top, controls=[controller_top],
                        width=view_width_top, height=view_height_top)

renderer_top

In [ ]:
# Animate the spinning top simulation
# Use smaller time steps for numerical stability with multiple constraints
from time import sleep
for i in range(10000):
    mss_top.simulate(0.01, 50)
    for m, mvis in zip(mss_top.masses, masses_top):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    constraintpos_top = []
    for c in mss_top.constraints:
        pA = mss_top[c.connectors[0]].pos
        pB = mss_top[c.connectors[1]].pos
        constraintpos_top.append([pA, pB])
    springs_top.geometry = LineSegmentsGeometry(positions=constraintpos_top)
    sleep(0.01)

# Katapult juhu!

ich bau jetzt ein Katapult

In [ ]:

mss_kat = MassSpringSystem3d()
mss_kat.gravity = (0, 0, -9.81)

fk1 = mss_kat.add(Fix((0, 0, 0)))
fk2 = mss_kat.add(Fix((2, 0, 0)))
fk3 = mss_kat.add(Fix((0, 2, 0)))
fk4 = mss_kat.add(Fix((2, 2, 0)))
pmk1 = (0, 1, 2)
pmk2 = (1, 1, 2)
pmk3 = (2, 1, 2)
pmk4 = (1, 1.1, 2.7)
pmk5 = (1, -0.5, 0.5)
pmk6 = (1, -1, 0.5)
pmk7 = (1, 1.5, 0.2)

mk1 = mss_kat.add(Mass(1, pmk1))
mk2 = mss_kat.add(Mass(1, pmk2))
mk3 = mss_kat.add(Mass(1, pmk3))
mk4 = mss_kat.add(Mass(10, pmk4))
mk5 = mss_kat.add(Mass(0.1, pmk5))
#mk6 = mss_kat.add(Mass(0.5, pmk6))
mk7 = mss_kat.add(Mass(1, pmk7))
import math

#kurz Längen berechnung
d1 = math.sqrt( (0-pmk1[0])**2 + (0-pmk1[1])**2 + (0-pmk1[2])**2 )        #Boden zu seitlicher Masse
d2 = math.sqrt( (0-pmk2[0])**2 + (0-pmk2[1])**2 + (0-pmk2[2])**2 )        #Boden zu mittlerer Masse
d3 = math.sqrt( (pmk2[0]-pmk4[0])**2 + (pmk2[1]-pmk4[1])**2 + (pmk2[2]-pmk4[2])**2 )      #Drehpunkt zu oberer Masse
d4 = math.sqrt( (pmk2[0]-pmk5[0])**2 + (pmk2[1]-pmk5[1])**2 + (pmk2[2]-pmk5[2])**2 )    #Drehpunkt zu unterer Masse
d5 = math.sqrt( (pmk4[0]-pmk5[0])**2 + (pmk4[1]-pmk5[1])**2 + (pmk4[2]-pmk5[2])**2 )  #obere zu unterer Masse
d6 = math.sqrt( (pmk1[0]-pmk2[0])**2 + (pmk1[1]-pmk2[1])**2 + (pmk1[2]-pmk2[2])**2 )  #seitliche Massen untereinander
#d7 = math.sqrt( (pmk5[0]-pmk6[0])**2 + (pmk5[1]-pmk6[1])**2 + (pmk5[2]-pmk6[2])**2 )  
d8 = math.sqrt( (pmk5[0]-pmk7[0])**2 + (pmk5[1]-pmk7[1])**2 + (pmk5[2]-pmk7[2])**2 )  

mss_kat.add(Spring(d1, 3000, (fk1, mk1)))
mss_kat.add(Spring(d1, 3000, (fk3, mk1)))
mss_kat.add(Spring(d1, 3000, (fk2, mk3)))
mss_kat.add(Spring(d1, 3000, (fk4, mk3)))

#3 constraints, 1 spring ist am stabilsten
mss_kat.add_constraint(DistanceConstraint(fk1, mk2, d2))
mss_kat.add_constraint(DistanceConstraint(fk2, mk2, d2))
mss_kat.add_constraint(DistanceConstraint(fk3, mk2, d2))
#mss_kat.add_constraint(DistanceConstraint(fk4, mk2, d2))
#mss_kat.add(Spring(d2, 5000, (fk1, mk2)))
#mss_kat.add(Spring(d2, 5000, (fk2, mk2)))
#mss_kat.add(Spring(d2, 5000, (fk3, mk2)))
mss_kat.add(Spring(d2, 5000, (fk4, mk2)))

temp_spring1 = mss_kat.add(Spring(0.3, 200, (fk3, mk4)))
temp_spring2 = mss_kat.add(Spring(0.3, 200, (fk4, mk4)))

mss_kat.add(Spring(d6, 10, (mk1, mk2)))
mss_kat.add(Spring(d6, 10, (mk3, mk2)))

mss_kat.add_constraint(DistanceConstraint(mk2, mk4, d3))
mss_kat.add_constraint(DistanceConstraint(mk2, mk5, d4))
mss_kat.add_constraint(DistanceConstraint(mk4, mk5, d5))
#mss_kat.add_constraint(DistanceConstraint(mk5, mk6, d7))
mss_kat.add_constraint(DistanceConstraint(mk5, mk7, d8))
#kleiner stupser
#mss_kat.set_mass_velocity(3, (0, -0.1, 0.1))

3

In [69]:
# Visualize it! *funkel*
masses_kat = []
for m in mss_kat.masses:
    masses_kat.append(
        Mesh(SphereBufferGeometry(0.1, 16, 16),
             MeshStandardMaterial(color='red'),
             position=m.pos))

fixes_kat = []
for f in mss_kat.fixes:
    fixes_kat.append(
        Mesh(SphereBufferGeometry(0.15, 32, 16),
             MeshStandardMaterial(color='blue'),
             position=f.pos))

constraintpos_kat = []
for c in mss_kat.constraints:
    pA = mss_kat[c.connectors[0]].pos
    pB = mss_kat[c.connectors[1]].pos
    constraintpos_kat.append([pA, pB])
springpos_kat = []
for s in mss_kat.springs:
    pA = mss_kat[s.connectors[0]].pos
    pB = mss_kat[s.connectors[1]].pos
    springpos_kat.append([pA, pB])

if constraintpos_kat:
    springgeo_kat = LineSegmentsGeometry(positions=constraintpos_kat)
    m2_kat = LineMaterial(linewidth=2, color='cyan')
    distance_lines = LineSegments2(springgeo_kat, m2_kat)
else:
    distance_lines = None

if springpos_kat:
    springgeo_spring = LineSegmentsGeometry(positions=springpos_kat)
    m2_spring = LineMaterial(linewidth=2, color='cyan') 
    spring_lines = LineSegments2(springgeo_spring, m2_spring)
else:
    spring_lines = None
axes_kat = AxesHelper(1)

In [70]:
# Render it!
view_width_kat = 600
view_height_kat = 400

camera_kat = PerspectiveCamera(position=[6, 5, 3], aspect=view_width_kat/view_height_kat)
key_light_kat = DirectionalLight(position=[0, 10, 10])
ambient_light_kat = AmbientLight()
camera_kat.up = [0, 0, 1]

# Sammle alle Objekte, inklusive der separaten Lines
scene_items = [*masses_kat, *fixes_kat, axes_kat, camera_kat, key_light_kat, ambient_light_kat]
if distance_lines:
    scene_items.append(distance_lines)
if spring_lines:
    scene_items.append(spring_lines)

scene_kat = Scene(children=scene_items)
controller_kat = OrbitControls(controlling=camera_kat)
renderer_kat = Renderer(camera=camera_kat, scene=scene_kat, controls=[controller_kat],
                        width=view_width_kat, height=view_height_kat)

renderer_kat

Renderer(camera=PerspectiveCamera(aspect=1.5, position=(6.0, 5.0, 3.0), projectionMatrix=(1.0, 0.0, 0.0, 0.0, …

In [71]:
# Studio Ghibli kann einpacken
from time import sleep
for i in range(10000):
    mss_kat.simulate(0.01, 100)
    for m, mvis in zip(mss_kat.masses, masses_kat):
        mvis.position = (m.pos[0], m.pos[1], m.pos[2])

    # DistanceConstraints aktualisieren
    constraintpos_kat = []
    for c in mss_kat.constraints:
        pA = mss_kat[c.connectors[0]].pos
        pB = mss_kat[c.connectors[1]].pos
        constraintpos_kat.append([pA, pB])
    if distance_lines:
        distance_lines.geometry = LineSegmentsGeometry(positions=constraintpos_kat)

    # Springs aktualisieren
    springpos_kat = []
    for s in mss_kat.springs:
        pA = mss_kat[s.connectors[0]].pos
        pB = mss_kat[s.connectors[1]].pos
        springpos_kat.append([pA, pB])
    if spring_lines:
        spring_lines.geometry = LineSegmentsGeometry(positions=springpos_kat)

    sleep(0.01)

KeyboardInterrupt: 